Enrollment No.:240840116005

Practical-9 Design and develop a simple interactive dashboard using Dash/Plotly for business reporting.

## Step 1 — Install required libraries

In [1]:
!pip -q install dash plotly pandas

## Step 2 — Import libraries

In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Step 3 — Create sample business data

In [3]:
data = {
    'Date': [
        '2026-01-05', '2026-01-12', '2026-02-08', '2026-02-15',
        '2026-03-10', '2026-03-20', '2026-04-05', '2026-04-18',
        '2026-05-08', '2026-05-22', '2026-06-10', '2026-06-25',
        '2026-07-05', '2026-07-19', '2026-08-09', '2026-08-21',
        '2026-09-10', '2026-09-25', '2026-10-05', '2026-10-20'
    ],
    'Category': [
        'Electronics', 'Clothing', 'Furniture', 'Electronics',
        'Clothing', 'Furniture', 'Electronics', 'Clothing',
        'Furniture', 'Electronics', 'Clothing', 'Furniture',
        'Electronics', 'Clothing', 'Furniture', 'Electronics',
        'Clothing', 'Furniture', 'Electronics', 'Clothing'
    ],
    'Region': [
        'North', 'South', 'East', 'West', 'North', 'South',
        'East', 'West', 'North', 'South', 'East', 'West',
        'North', 'South', 'East', 'West', 'North', 'South',
        'East', 'West'
    ],
    'Sales': [
        12000, 8000, 15000, 18000, 10000, 14000, 22000,
        12000, 17000, 25000, 16000, 19000, 28000, 18000,
        21000, 30000, 20000, 24000, 32000, 22000
    ],
    'Profit': [
        3000, 2000, 4000, 5000, 2500, 3500, 6000, 3000,
        4500, 7000, 4000, 5000, 7500, 4500, 5500, 8000,
        5000, 6000, 8500, 5500
    ],
    'Units': [
        12, 15, 10, 18, 20, 14, 22, 16, 19, 25,
        21, 18, 28, 24, 20, 30, 26, 22, 32, 28
    ]
}

df = pd.DataFrame(data)
df['Date'] = pd.to_datetime(df['Date'])
df['Order ID'] = range(1, len(df) + 1)

print("Dataset preview:")
display(df.head())
print("Number of records:", len(df))

Dataset preview:


,Date,Category,Region,Sales,Profit,Units,Order ID
0,2026-01-05,Electronics,North,12000,3000,12,1
1,2026-01-12,Clothing,South,8000,2000,15,2
2,2026-02-08,Furniture,East,15000,4000,10,3
3,2026-02-15,Electronics,West,18000,5000,18,4
4,2026-03-10,Clothing,North,10000,2500,20,5


Number of records: 20


## Step 4 — Check the dataset and business totals

In [4]:
print("Dataset shape:", df.shape)
print("\\nDataset information:")
df.info()

print("\\nTotal sales: ₹{:,.0f}".format(df['Sales'].sum()))
print("Total profit: ₹{:,.0f}".format(df['Profit'].sum()))
print("Total orders:", len(df))

Dataset shape: (20, 7)
\nDataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      20 non-null     datetime64[ns]
 1   Category  20 non-null     object        
 2   Region    20 non-null     object        
 3   Sales     20 non-null     int64         
 4   Profit    20 non-null     int64         
 5   Units     20 non-null     int64         
 6   Order ID  20 non-null     int64         
dtypes: datetime64[ns](1), int64(4), object(2)
memory usage: 1.2+ KB
\nTotal sales: ₹383,000
Total profit: ₹100,000
Total orders: 20


## Step 5 — Initialize the Dash application

In [5]:
app = Dash(__name__)
app.title = "Business Reporting Dashboard"

card_style = {
    'backgroundColor': 'white',
    'padding': '15px',
    'width': '32%',
    'textAlign': 'center',
    'borderRadius': '10px',
    'boxShadow': '0 2px 8px rgba(0,0,0,0.08)'
}

chart_style = {'width': '49%', 'display': 'inline-block', 'verticalAlign': 'top'}

## Step 6 — Design the dashboard layout

In [6]:
region_options = [{'label': 'All Regions', 'value': 'All'}] + [
    {'label': r, 'value': r} for r in sorted(df['Region'].unique())
]
category_options = [{'label': 'All Categories', 'value': 'All'}] + [
    {'label': c, 'value': c} for c in sorted(df['Category'].unique())
]

app.layout = html.Div(
    style={
        'backgroundColor': '#f4f6f9',
        'padding': '20px',
        'fontFamily': 'Arial',
        'minHeight': '100vh'
    },
    children=[
        html.H1(
            "Business Reporting Dashboard",
            style={'textAlign': 'center', 'color': '#1f2937'}
        ),
        html.P(
            "Interactive Sales and Profit Analysis",
            style={'textAlign': 'center', 'color': '#555'}
        ),
        html.Hr(),

        html.Div([
            html.Div([
                html.Label("Select Region"),
                dcc.Dropdown(
                    id='region-filter',
                    options=region_options,
                    value='All',
                    clearable=False
                )
            ], style={'width': '48%'}),

            html.Div([
                html.Label("Select Category"),
                dcc.Dropdown(
                    id='category-filter',
                    options=category_options,
                    value='All',
                    clearable=False
                )
            ], style={'width': '48%'})
        ], style={
            'display': 'flex',
            'justifyContent': 'space-between',
            'gap': '15px',
            'marginBottom': '25px'
        }),

        html.Div([
            html.Div([
                html.H4("Total Sales"),
                html.H2(id='total-sales')
            ], style=card_style),

            html.Div([
                html.H4("Total Profit"),
                html.H2(id='total-profit')
            ], style=card_style),

            html.Div([
                html.H4("Total Orders"),
                html.H2(id='total-orders')
            ], style=card_style)
        ], style={
            'display': 'flex',
            'justifyContent': 'space-between',
            'gap': '15px',
            'marginBottom': '25px'
        }),

        html.Div([
            dcc.Graph(id='sales-chart', style=chart_style),
            dcc.Graph(id='category-chart', style=chart_style)
        ]),

        html.Div([
            dcc.Graph(id='region-chart', style=chart_style),
            dcc.Graph(id='profit-chart', style=chart_style)
        ])
    ]
)

## Step 7 — Create four interactive charts and callback

In [7]:
@app.callback(
    [
        Output('sales-chart', 'figure'),
        Output('category-chart', 'figure'),
        Output('region-chart', 'figure'),
        Output('profit-chart', 'figure'),
        Output('total-sales', 'children'),
        Output('total-profit', 'children'),
        Output('total-orders', 'children')
    ],
    [
        Input('region-filter', 'value'),
        Input('category-filter', 'value')
    ]
)
def update_dashboard(selected_region, selected_category):
    # Apply selected filters
    filtered_df = df.copy()

    if selected_region != 'All':
        filtered_df = filtered_df[filtered_df['Region'] == selected_region]

    if selected_category != 'All':
        filtered_df = filtered_df[filtered_df['Category'] == selected_category]

    # Chart 1: Monthly Sales Trend (Line chart)
    monthly_sales = (
        filtered_df.groupby(filtered_df['Date'].dt.to_period('M'))['Sales']
        .sum().reset_index()
    )
    monthly_sales['Month'] = monthly_sales['Date'].astype(str)

    fig1 = px.line(
        monthly_sales,
        x='Month',
        y='Sales',
        markers=True,
        title='1. Monthly Sales Trend',
        template='plotly_white'
    )
    fig1.update_traces(line=dict(width=3), marker=dict(size=9))
    fig1.update_layout(yaxis_title='Sales (₹)', xaxis_title='Month')

    # Chart 2: Sales by Category (Bar chart)
    category_sales = (
        filtered_df.groupby('Category', as_index=False)['Sales'].sum()
    )
    fig2 = px.bar(
        category_sales,
        x='Category',
        y='Sales',
        color='Category',
        title='2. Sales by Category',
        template='plotly_white',
        text_auto=True
    )
    fig2.update_layout(yaxis_title='Sales (₹)')

    # Chart 3: Regional Sales Distribution (Doughnut chart)
    region_sales = (
        filtered_df.groupby('Region', as_index=False)['Sales'].sum()
    )
    fig3 = px.pie(
        region_sales,
        names='Region',
        values='Sales',
        title='3. Regional Sales Distribution',
        hole=0.4
    )

    # Chart 4: Sales vs Profit (Scatter chart)
    fig4 = px.scatter(
        filtered_df,
        x='Sales',
        y='Profit',
        color='Category',
        size='Units',
        hover_data=['Region', 'Order ID'],
        title='4. Sales vs Profit Analysis',
        template='plotly_white'
    )
    fig4.update_layout(xaxis_title='Sales (₹)', yaxis_title='Profit (₹)')

    # KPI calculations
    total_sales = filtered_df['Sales'].sum()
    total_profit = filtered_df['Profit'].sum()
    total_orders = len(filtered_df)

    return (
        fig1,
        fig2,
        fig3,
        fig4,
        f"₹{total_sales:,.0f}",
        f"₹{total_profit:,.0f}",
        str(total_orders)
    )

## Step 8 — Run the dashboard in Google Colab

In [ ]:
app.run(
    jupyter_mode='inline',
    debug=False,
    port=8050
)